In [ ]:

from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
model = "gemini-2.5-flash-lite"
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "parts": [{"text": text}]
    })

def add_assistant_message(messages, text):
    messages.append({
        "role": "model",
        "parts": [{"text": text}]
    })

def chatwithstreamfunction(messages, system_prompt=None, temperature=1.0 , stop_sequences=[]):
    config = {
        "max_output_tokens": 100000,
        "temperature": temperature
    }

    if system_prompt:
        config["system_instruction"] = system_prompt

    if stop_sequences:
        config["stop_sequences"]  = stop_sequences
   
    response = client.models.generate_content(
        model=model,
        contents=messages
        config=types.GenerateContentConfig(**config)
    )
    return response.text

In [6]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format" : "json" or "python" or "regex"
  }
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chatwithstreamfunction(messages, stop_sequences=["```"])
    print(text)
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text.strip())


In [7]:
# here will generate task dataset using gemini
dataset  = generate_dataset()
# here we create dataset.json file 
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)
# print(dataset)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 35.375763054s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash-lite', 'location': 

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
* response only with python ,JSON or a plan Regex
* do not add any comments or commentary or explanation
""" 
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chatwithstreamfunction(messages, stop_sequences=["```"])
    return output

In [ ]:
# here we are giving grader to dataset using model 
def grade_by_model(task, solution):

    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chatwithstreamfunction(messages, stop_sequences=["```"])
    print("RAW RESPONSE:")
    print(eval_text)
    eval_text = eval_text.replace("```json", "").replace("```", "").strip()
    print("CLEANED RESPONSE:")
    print(eval_text)
    return json.loads(eval_text)


In [ ]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response , test_case):
    format = test_case["formate"]
    if format =="json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output= run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    score = (model_score+ syntax_score)/2
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [8]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results=[]
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score  = mean([result["score"] for result in results])
    print(f"average score:{average_score}")
    return results
with open("dataset.json", 'r') as f:
    dataset = json.load(f)
results = run_eval(dataset)

NameError: name 'run_test_case' is not defined

In [ ]:
with open("dataset.json", 'r') as f:
    dataset = json.load(f)
results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent =2))